# EfficientNet-B2 B2-4 + Green CLAHE Background Subtraction Full Run

This notebook performs one full confirmation run using the exact previous **B2-4** configuration.
The only changed factor is preprocessing: `green_clahe_bgsub_mask`.

- EfficientNet-B2 with ImageNet weights
- full backbone fine-tuning
- custom 256-unit classification head
- cross-entropy loss
- no augmentation
- 60 configured epochs, patience 12
- checkpoint selection by validation Macro F1
- official test set evaluated only after loading the best validation checkpoint


## 1. Setup And Configuration

In [ ]:
import gc
import json
import random
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

try:
    from skimage.exposure import match_histograms
except Exception:
    match_histograms = None

In [ ]:
# ============================================================
# Paths and reproducibility
# ============================================================
DATA_ROOT = Path("/kaggle/input/datasets/paulconrad21/eye-disease-dataset/eye-disease-data")
TRAIN_CSV_PATH = DATA_ROOT / "2. Groundtruths" / "a. IDRiD_Disease Grading_Training Labels.csv"
TEST_CSV_PATH = DATA_ROOT / "2. Groundtruths" / "b. IDRiD_Disease Grading_Testing Labels.csv"
TRAIN_IMAGE_DIR = DATA_ROOT / "1. Original Images" / "a. Training Set"
TEST_IMAGE_DIR = DATA_ROOT / "1. Original Images" / "b. Testing Set"

IMAGE_COL = "Image name"
LABEL_COL = "Retinopathy grade"
NUM_CLASSES = 5
IMAGE_SIZE = 384
VAL_SIZE = 0.2
SEED = 42
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# Two-stage experiment budget
# ============================================================
RUN_SCREENING = True
RUN_CONFIRMATION = True
SCREENING_EPOCHS = 25
SCREENING_PATIENCE = 6
CONFIRMATION_EPOCHS = 60
CONFIRMATION_PATIENCE = 12
TOP_N_NEW_METHODS = 2
MIN_DELTA = 1e-4

# These compound recipes are the genuinely new additions from the partner notebook.
# "none" is included as the B2-4 control.
SCREENING_PREPROCESSING_MODES = [
    "none",
    "gamma_correction",
    "gray_gamma_clahe",
    "green_histmatch_median_clahe_unsharp_crop",
    "green_clahe_bgsub_mask",
    "clean_green_masked",
    "white_balance",
    "white_balance_clahe",
]

# The partner notebook also contains these simpler modes. They already overlap with
# preprocessing available in the previous CNN pipeline, so they are optional here.
INCLUDE_BASIC_PARTNER_MODES = False
OPTIONAL_BASIC_MODES = ["green_channel", "clahe_lab"]
if INCLUDE_BASIC_PARTNER_MODES:
    SCREENING_PREPROCESSING_MODES += OPTIONAL_BASIC_MODES

RESULTS_CSV = "/kaggle/working/b2_4_green_clahe_bgsub_full_results.csv"
EPOCH_HISTORY_CSV = "/kaggle/working/b2_4_green_clahe_bgsub_full_epoch_history.csv"
SELECTED_METHODS_CSV = "/kaggle/working/b2_4_green_clahe_bgsub_full_run_config.csv"

# Exact fixed configurations of the previous three strongest individual models.
MODEL_CONFIGS = {
    "B2-4": {
        "loss_name": "cross_entropy",
        "use_augmentation": False,
        "report_note": "Original B2-4 configuration: CE, no augmentation, full fine-tuning.",
    },
    "B2-5": {
        "loss_name": "cross_entropy",
        "use_augmentation": True,
        "report_note": "Original B2-5 configuration: CE, mild augmentation, full fine-tuning.",
    },
    "B2-6": {
        "loss_name": "weighted_cross_entropy",
        "use_augmentation": False,
        "report_note": "Original B2-6 configuration: weighted CE, no augmentation, full fine-tuning.",
    },
}

# Previous baseline rows copied from Output Performance/all_experiment_results_combined.csv.
BASELINE_RESULTS = {
    "B2-4": {"val_macro_f1": 0.576730, "test_macro_f1": 0.515671, "test_kappa": 0.600727},
    "B2-5": {"val_macro_f1": 0.657509, "test_macro_f1": 0.497557, "test_kappa": 0.692956},
    "B2-6": {"val_macro_f1": 0.629050, "test_macro_f1": 0.487650, "test_kappa": 0.646306},
}

FIXED_TRAINING_CONFIG = {
    "batch_size": 4,
    "optimizer_name": "adam",
    "learning_rate": 5e-5,
    "weight_decay": 1e-4,
    "model_dropout": 0.20,
    "custom_head_hidden_dim": 256,
    "use_frozen_backbone": False,
    "selection_metric": "macro_f1",
}

print(f"Device: {DEVICE}")
print(f"Screening methods ({len(SCREENING_PREPROCESSING_MODES)}): {SCREENING_PREPROCESSING_MODES}")
print(f"Confirmation models: {list(MODEL_CONFIGS)}")

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

for required_path in [TRAIN_CSV_PATH, TEST_CSV_PATH, TRAIN_IMAGE_DIR, TEST_IMAGE_DIR]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required path not found: {required_path}")

train_full_df = pd.read_csv(TRAIN_CSV_PATH)[[IMAGE_COL, LABEL_COL]].copy()
test_df = pd.read_csv(TEST_CSV_PATH)[[IMAGE_COL, LABEL_COL]].copy()
train_full_df[LABEL_COL] = train_full_df[LABEL_COL].astype(int)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)

print("Training distribution:")
display(train_full_df[LABEL_COL].value_counts().sort_index())
print("Official test distribution:")
display(test_df[LABEL_COL].value_counts().sort_index())

## 2. Partner Preprocessing Recipes

In [ ]:
def resolve_image_path(image_dir, image_name):
    image_dir = Path(image_dir)
    image_name = str(image_name)
    direct = image_dir / image_name
    if direct.exists():
        return direct
    for extension in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]:
        candidate = image_dir / f"{image_name}{extension}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {image_name} in {image_dir}")


class PartnerFundusPreprocessor:
    def __init__(self, mode="none", gamma=1.2, reference_green=None):
        self.mode = mode
        self.gamma = gamma
        self.reference_green = reference_green

    def apply_gamma_rgb(self, image):
        inverse_gamma = 1.0 / self.gamma
        table = np.array([((value / 255.0) ** inverse_gamma) * 255 for value in range(256)]).astype("uint8")
        return cv2.LUT(image, table)

    @staticmethod
    def apply_clahe_gray(gray, clip_limit=1.5):
        return cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8)).apply(gray)

    @staticmethod
    def apply_clahe_lab(image, clip_limit=1.0):
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        lightness, a_channel, b_channel = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(cv2.merge((lightness, a_channel, b_channel)), cv2.COLOR_LAB2RGB)

    @staticmethod
    def white_balance(image):
        balanced = image.astype(np.float32)
        averages = np.maximum(balanced.mean(axis=(0, 1)), 1e-6)
        gray_average = averages.mean()
        balanced *= gray_average / averages
        return np.clip(balanced, 0, 255).astype(np.uint8)

    def histogram_match(self, gray):
        if match_histograms is not None and self.reference_green is not None:
            matched = match_histograms(gray, self.reference_green, channel_axis=None)
            return np.clip(matched, 0, 255).astype(np.uint8)
        return cv2.equalizeHist(gray)

    @staticmethod
    def remove_black_corners(gray, threshold=8):
        coordinates = np.argwhere(gray > threshold)
        if coordinates.size == 0:
            return gray
        y0, x0 = coordinates.min(axis=0)
        y1, x1 = coordinates.max(axis=0) + 1
        return gray[y0:y1, x0:x1]

    @staticmethod
    def green_background_subtraction(image, sigma):
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        green = image[:, :, 1]
        enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(green)
        background = cv2.GaussianBlur(enhanced, (0, 0), sigmaX=sigma)
        equalized = cv2.addWeighted(enhanced, 4, background, -4, 128)
        cleaned = cv2.bitwise_and(equalized, equalized, mask=mask)
        return np.stack([cleaned, cleaned, cleaned], axis=-1)

    def __call__(self, image):
        array = np.array(image.convert("RGB"))

        if self.mode == "none":
            result = array
        elif self.mode == "green_channel":
            green = array[:, :, 1]
            result = np.stack([green, green, green], axis=-1)
        elif self.mode == "clahe_lab":
            result = self.apply_clahe_lab(array, clip_limit=1.0)
        elif self.mode == "gamma_correction":
            result = self.apply_gamma_rgb(array)
        elif self.mode == "gray_gamma_clahe":
            gray = cv2.cvtColor(array, cv2.COLOR_RGB2GRAY)
            gray_rgb = np.stack([gray, gray, gray], axis=-1)
            gamma_gray = self.apply_gamma_rgb(gray_rgb)[:, :, 0]
            enhanced = self.apply_clahe_gray(gamma_gray, clip_limit=1.5)
            result = np.stack([enhanced, enhanced, enhanced], axis=-1)
        elif self.mode == "green_histmatch_median_clahe_unsharp_crop":
            green = self.histogram_match(array[:, :, 1])
            green = cv2.medianBlur(green, ksize=3)
            green = self.apply_clahe_gray(green, clip_limit=1.5)
            blurred = cv2.GaussianBlur(green, (0, 0), 1.0)
            green = cv2.addWeighted(green, 2.5, blurred, -1.5, 0)
            green = self.remove_black_corners(np.clip(green, 0, 255).astype(np.uint8))
            result = np.stack([green, green, green], axis=-1)
        elif self.mode == "green_clahe_bgsub_mask":
            result = self.green_background_subtraction(array, sigma=15)
        elif self.mode == "clean_green_masked":
            result = self.green_background_subtraction(array, sigma=30)
        elif self.mode == "white_balance":
            result = self.white_balance(array)
        elif self.mode == "white_balance_clahe":
            result = self.apply_clahe_lab(self.white_balance(array), clip_limit=1.5)
        else:
            raise ValueError(f"Unknown preprocessing mode: {self.mode}")

        return Image.fromarray(result.astype(np.uint8))


class FundusDataset(Dataset):
    def __init__(self, dataframe, image_dir, preprocessing_mode, reference_green, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.preprocessor = PartnerFundusPreprocessor(preprocessing_mode, reference_green=reference_green)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(resolve_image_path(self.image_dir, row[IMAGE_COL])).convert("RGB")
        image = self.preprocessor(image)
        image = self.transform(image)
        return image, int(row[LABEL_COL])

In [ ]:
def build_transform(train=False, use_augmentation=False):
    steps = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
    if train and use_augmentation:
        steps.extend([
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05, hue=0.02),
        ])
    steps.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return transforms.Compose(steps)


def compute_class_weights(labels):
    counts = np.maximum(np.bincount(np.asarray(labels), minlength=NUM_CLASSES), 1)
    return torch.tensor(len(labels) / (NUM_CLASSES * counts), dtype=torch.float32)


def create_dataloaders(run_config):
    train_df, val_df = train_test_split(
        train_full_df,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=train_full_df[LABEL_COL],
    )

    reference_path = resolve_image_path(TRAIN_IMAGE_DIR, train_df.iloc[0][IMAGE_COL])
    reference_green = np.array(Image.open(reference_path).convert("RGB"))[:, :, 1]
    preprocessing_mode = run_config["preprocessing_mode"]
    use_augmentation = run_config["use_augmentation"]

    train_dataset = FundusDataset(
        train_df, TRAIN_IMAGE_DIR, preprocessing_mode, reference_green,
        build_transform(train=True, use_augmentation=use_augmentation),
    )
    val_dataset = FundusDataset(
        val_df, TRAIN_IMAGE_DIR, preprocessing_mode, reference_green,
        build_transform(train=False),
    )
    test_dataset = FundusDataset(
        test_df, TEST_IMAGE_DIR, preprocessing_mode, reference_green,
        build_transform(train=False),
    )

    loader_kwargs = {
        "batch_size": FIXED_TRAINING_CONFIG["batch_size"],
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE.type == "cuda",
    }
    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

    return train_loader, val_loader, test_loader, compute_class_weights(train_df[LABEL_COL].values), train_df, val_df


def show_preprocessing_examples(modes=None, image_index=0):
    modes = modes or SCREENING_PREPROCESSING_MODES
    image_path = resolve_image_path(TRAIN_IMAGE_DIR, train_full_df.iloc[image_index][IMAGE_COL])
    original = Image.open(image_path).convert("RGB")
    reference_green = np.array(original)[:, :, 1]

    import matplotlib.pyplot as plt
    figure, axes = plt.subplots(len(modes), 2, figsize=(10, 4 * len(modes)))
    for row, mode in enumerate(modes):
        processed = PartnerFundusPreprocessor(mode, reference_green=reference_green)(original)
        axes[row, 0].imshow(original)
        axes[row, 0].set_title("Original")
        axes[row, 1].imshow(processed)
        axes[row, 1].set_title(mode)
        axes[row, 0].axis("off")
        axes[row, 1].axis("off")
    plt.tight_layout()
    plt.show()


# Visual QA is optional because the figure can be large.
SHOW_PREPROCESSING_EXAMPLES = False
if SHOW_PREPROCESSING_EXAMPLES:
    show_preprocessing_examples()

## 3. EfficientNet-B2 And Training Utilities

In [ ]:
def create_model():
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
    in_features = model.classifier[-1].in_features
    hidden_dim = FIXED_TRAINING_CONFIG["custom_head_hidden_dim"]
    model.classifier = nn.Sequential(
        nn.Linear(in_features, hidden_dim),
        nn.SiLU(inplace=True),
        nn.BatchNorm1d(hidden_dim),
        nn.Dropout(FIXED_TRAINING_CONFIG["model_dropout"]),
        nn.Linear(hidden_dim, NUM_CLASSES),
    )
    return model.to(DEVICE)


def create_loss(loss_name, class_weights):
    if loss_name == "cross_entropy":
        return nn.CrossEntropyLoss()
    if loss_name == "weighted_cross_entropy":
        return nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    raise ValueError(f"Unsupported loss: {loss_name}")


def compute_metrics(labels, predictions):
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "kappa": cohen_kappa_score(labels, predictions, weights="quadratic"),
    }


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    predictions, labels_all = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            running_loss += criterion(logits, labels).item() * images.size(0)
            predictions.extend(logits.argmax(dim=1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    return running_loss / len(loader.dataset), compute_metrics(labels_all, predictions), predictions, labels_all


def flatten_report(prefix, labels, predictions):
    report = classification_report(
        labels, predictions, labels=list(range(NUM_CLASSES)),
        output_dict=True, zero_division=0,
    )
    output = {}
    for class_index in range(NUM_CLASSES):
        class_result = report[str(class_index)]
        output[f"{prefix}_precision_class_{class_index}"] = class_result["precision"]
        output[f"{prefix}_recall_class_{class_index}"] = class_result["recall"]
        output[f"{prefix}_f1_class_{class_index}"] = class_result["f1-score"]
        output[f"{prefix}_support_class_{class_index}"] = class_result["support"]
    output[f"{prefix}_weighted_f1"] = report["weighted avg"]["f1-score"]
    return output

## 4. Full B2-4 Confirmation Run

The complete B2-4 configuration is trained once with `green_clahe_bgsub_mask` preprocessing.


In [ ]:
def save_progress(summary_rows, epoch_rows):
    pd.DataFrame(summary_rows).to_csv(RESULTS_CSV, index=False)
    pd.DataFrame(epoch_rows).to_csv(EPOCH_HISTORY_CSV, index=False)


def run_experiment(model_id, preprocessing_mode, stage, num_epochs, patience):
    model_config = MODEL_CONFIGS[model_id]
    run_id = f"{stage}_{model_id}_{preprocessing_mode}"
    print("\n" + "=" * 90)
    print(f"STARTING {run_id}")
    print("=" * 90)

    set_seed()
    run_config = {
        **FIXED_TRAINING_CONFIG,
        **model_config,
        "model_id": model_id,
        "preprocessing_mode": preprocessing_mode,
        "stage": stage,
        "num_epochs": num_epochs,
        "patience": patience,
    }

    train_loader, val_loader, test_loader, class_weights, train_df, val_df = create_dataloaders(run_config)
    model = create_model()
    criterion = create_loss(run_config["loss_name"], class_weights)
    optimizer = optim.Adam(
        model.parameters(),
        lr=run_config["learning_rate"],
        weight_decay=run_config["weight_decay"],
    )

    checkpoint_path = f"/kaggle/working/{run_id.lower()}_best_efficientnet_b2.pth"
    best_score, best_epoch, epochs_without_improvement = -1.0, 0, 0
    epoch_rows = []
    start_time = time.time()

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_metrics, _, _ = evaluate(model, val_loader, criterion)
        current_score = val_metrics["macro_f1"]

        epoch_rows.append({
            "run_id": run_id,
            "stage": stage,
            "model_id": model_id,
            "preprocessing_mode": preprocessing_mode,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_kappa": val_metrics["kappa"],
        })

        print(
            f"{run_id} | Epoch {epoch}/{num_epochs} | Train Loss {train_loss:.4f} | "
            f"Val Macro F1 {val_metrics['macro_f1']:.4f} | "
            f"Val Kappa {val_metrics['kappa']:.4f}"
        )

        if current_score > best_score + MIN_DELTA:
            best_score = current_score
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    val_loss, val_metrics, val_predictions, val_labels = evaluate(model, val_loader, criterion)
    # The official test set stays untouched during screening. Only methods selected
    # automatically from validation Macro F1 reach the confirmation/test stage.
    if stage == "confirm":
        test_loss, test_metrics, test_predictions, test_labels = evaluate(model, test_loader, criterion)
    else:
        test_loss = np.nan
        test_metrics = {"accuracy": np.nan, "macro_f1": np.nan, "kappa": np.nan}
        test_predictions, test_labels = [], []
    baseline = BASELINE_RESULTS[model_id]

    summary = {
        "run_id": run_id,
        "stage": stage,
        "model_id": model_id,
        "model": "efficientnet_b2",
        "preprocessing_mode": preprocessing_mode,
        "best_model_path": checkpoint_path,
        "best_epoch": best_epoch,
        "epochs_ran": len(epoch_rows),
        "elapsed_minutes": (time.time() - start_time) / 60,
        "selection_metric": "macro_f1",
        "best_selection_score": best_score,
        "image_size": IMAGE_SIZE,
        "batch_size": run_config["batch_size"],
        "seed": SEED,
        "val_size": VAL_SIZE,
        "optimizer_name": run_config["optimizer_name"],
        "learning_rate": run_config["learning_rate"],
        "weight_decay": run_config["weight_decay"],
        "loss_name": run_config["loss_name"],
        "use_augmentation": run_config["use_augmentation"],
        "use_frozen_backbone": run_config["use_frozen_backbone"],
        "custom_head_hidden_dim": run_config["custom_head_hidden_dim"],
        "model_dropout": run_config["model_dropout"],
        "train_samples": len(train_df),
        "val_samples": len(val_df),
        "test_samples": len(test_df),
        "val_loss": val_loss,
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_kappa": val_metrics["kappa"],
        "test_loss": test_loss,
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_kappa": test_metrics["kappa"],
        "baseline_val_macro_f1": baseline["val_macro_f1"],
        "baseline_test_macro_f1": baseline["test_macro_f1"],
        "baseline_test_kappa": baseline["test_kappa"],
        # Deltas are only meaningful after the full-budget confirmation stage.
        "delta_val_macro_f1_vs_baseline": (
            val_metrics["macro_f1"] - baseline["val_macro_f1"] if stage == "confirm" else np.nan
        ),
        "delta_test_macro_f1_vs_baseline": (
            test_metrics["macro_f1"] - baseline["test_macro_f1"] if stage == "confirm" else np.nan
        ),
        "delta_test_kappa_vs_baseline": (
            test_metrics["kappa"] - baseline["test_kappa"] if stage == "confirm" else np.nan
        ),
        "val_confusion_matrix_json": json.dumps(confusion_matrix(val_labels, val_predictions, labels=list(range(NUM_CLASSES))).tolist()),
        "test_confusion_matrix_json": (
            json.dumps(confusion_matrix(test_labels, test_predictions, labels=list(range(NUM_CLASSES))).tolist())
            if stage == "confirm" else ""
        ),
        "report_note": model_config["report_note"],
    }
    summary.update(flatten_report("val", val_labels, val_predictions))
    if stage == "confirm":
        summary.update(flatten_report("test", test_labels, test_predictions))
    else:
        for class_index in range(NUM_CLASSES):
            summary[f"test_precision_class_{class_index}"] = np.nan
            summary[f"test_recall_class_{class_index}"] = np.nan
            summary[f"test_f1_class_{class_index}"] = np.nan
            summary[f"test_support_class_{class_index}"] = np.nan
        summary["test_weighted_f1"] = np.nan

    print(
        f"FINISHED {run_id}: Val Macro F1={val_metrics['macro_f1']:.4f}, "
        f"Test Macro F1={test_metrics['macro_f1']:.4f}, Test Kappa={test_metrics['kappa']:.4f}"
    )

    del model, optimizer, train_loader, val_loader, test_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return summary, epoch_rows

In [ ]:
# ============================================================
# Full B2-4 + green_clahe_bgsub_mask confirmation run
# ============================================================
TARGET_MODEL_ID = "B2-4"
TARGET_PREPROCESSING_MODE = "green_clahe_bgsub_mask"

run_config_df = pd.DataFrame([{
    "model_id": TARGET_MODEL_ID,
    "preprocessing_mode": TARGET_PREPROCESSING_MODE,
    "num_epochs": CONFIRMATION_EPOCHS,
    "patience": CONFIRMATION_PATIENCE,
    **FIXED_TRAINING_CONFIG,
    **MODEL_CONFIGS[TARGET_MODEL_ID],
}])
run_config_df.to_csv(SELECTED_METHODS_CSV, index=False)
display(run_config_df)

summary, epoch_rows = run_experiment(
    model_id=TARGET_MODEL_ID,
    preprocessing_mode=TARGET_PREPROCESSING_MODE,
    stage="confirm",
    num_epochs=CONFIRMATION_EPOCHS,
    patience=CONFIRMATION_PATIENCE,
)

results_df = pd.DataFrame([summary])
epoch_history_df = pd.DataFrame(epoch_rows)
results_df.to_csv(RESULTS_CSV, index=False)
epoch_history_df.to_csv(EPOCH_HISTORY_CSV, index=False)

print("\nFULL RUN RESULT")
display(results_df[[
    "model_id",
    "preprocessing_mode",
    "best_epoch",
    "epochs_ran",
    "val_accuracy",
    "val_macro_f1",
    "val_kappa",
    "test_accuracy",
    "test_macro_f1",
    "test_kappa",
    "delta_val_macro_f1_vs_baseline",
    "delta_test_macro_f1_vs_baseline",
    "delta_test_kappa_vs_baseline",
    "best_model_path",
]])

print("\nTest per-class F1:")
display(results_df[[
    "test_f1_class_0",
    "test_f1_class_1",
    "test_f1_class_2",
    "test_f1_class_3",
    "test_f1_class_4",
]])

try:
    from IPython.display import FileLink
    display(FileLink(RESULTS_CSV))
    display(FileLink(EPOCH_HISTORY_CSV))
    display(FileLink(SELECTED_METHODS_CSV))
except Exception:
    print(RESULTS_CSV)
    print(EPOCH_HISTORY_CSV)
    print(SELECTED_METHODS_CSV)